# Thí nghiệm CNN–LSTM song song

Mục tiêu của notebook là so sánh công bằng với `cnn_lstm/CNN_LSTM.ipynb`:

- Dùng đúng pipeline dữ liệu trong `preprocessing/preprocess_data.py`.
- Dùng cùng split 72% train / 8% validation / 20% test, seed 42.
- Dùng cùng tokenizer char-level, `MAX_LEN=1024` và bộ siêu tham số.
- Chỉ thay đổi topology: CNN và LSTM chạy **song song** từ cùng Embedding, sau đó concatenate.

Nhánh CNN giữ `Conv1D(128,k=3) → MaxPool(4) → Conv1D(128,k=5) → MaxPool(4)`.
Nhánh LSTM giữ `LSTM(128)`. `GlobalMaxPooling1D` là phép gom chuỗi bắt buộc để đầu ra
CNN có thể nối với vector đầu ra của LSTM.


## 1. Thiết lập môi trường và import

In [1]:
from pathlib import Path
import json
import os
import pickle
import random
import sys

START_DIR = Path.cwd().resolve()
if (START_DIR / "preprocessing" / "preprocess_data.py").exists():
    PROJECT_ROOT = START_DIR
elif START_DIR.name == "cnn_lstm_parallel" and (START_DIR.parent / "preprocessing" / "preprocess_data.py").exists():
    PROJECT_ROOT = START_DIR.parent
else:
    raise FileNotFoundError("Hãy chạy notebook từ repo root hoặc thư mục cnn_lstm_parallel.")

NOTEBOOK_DIR = PROJECT_ROOT / "cnn_lstm_parallel"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(NOTEBOOK_DIR)

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import (
    Concatenate, Conv1D, Dense, Dropout, Embedding,
    GlobalMaxPooling1D, Input, LSTM, MaxPooling1D,
)
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from preprocessing import preprocess_data as prep

SEED = 42
MAX_LEN = 1024
EMBEDDING_DIM = 64
BATCH_SIZE = 128
EPOCHS = 50

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


Project root: C:\Users\admin\Desktop\obfuscated-web-attack-detection
TensorFlow: 2.21.0
GPU available: False


## 2. Tiền xử lý dùng chung từ `preprocess_data.py`

In [ ]:
KAGGLE_PATH = PROJECT_ROOT / "SQLInjection_XSS_MixDataset.1.0.0.csv"
CSIC_PATH = PROJECT_ROOT / "csic_database.csv"
OBFUSCATION_PATH = PROJECT_ROOT / "obfuscation_dataset_full.xlsx"

# Dùng trực tiếp các hàm tiền xử lý đã có trong preprocessing/preprocess_data.py.
base_df = prep.clean(
    pd.concat(
        [prep.load_kaggle(str(KAGGLE_PATH)), prep.load_csic(str(CSIC_PATH))],
        ignore_index=True,
    ),
    deduplicate=True,
    drop_label_conflicts=True,
)
base_df = base_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Module chưa có hàm split nên phần chia dữ liệu được đặt trong notebook.
train_val_df, test_df = train_test_split(
    base_df, test_size=0.2, random_state=SEED, stratify=base_df["label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1, random_state=SEED, stratify=train_val_df["label"]
)
obfuscated_df = prep.clean(
    prep.load_obfuscation(str(OBFUSCATION_PATH)),
    deduplicate=True,
    drop_label_conflicts=False,
)


preprocessing_metadata = {
    "preprocessing_policy": {
        "source": "preprocessing/preprocess_data.py",
        "url_decode": False,
        "html_unescape": False,
        "lowercase": False,
        "whitespace_normalization_only": True,
        "tokenizer_rule": "Fit tokenizer on train split only.",
    },
    "splits": {
        "base_clean": prep.summarize(base_df),
        "train": prep.summarize(train_df),
        "val": prep.summarize(val_df),
        "test": prep.summarize(test_df),
        "obfuscated_test": prep.summarize(obfuscated_df),
    },
}

# preprocess_data.py chưa có tokenizer/vectorization nên định nghĩa phần này trong notebook.
tokenizer = Tokenizer(
    char_level=True,
    lower=False,
    filters="",
    oov_token="<OOV>",
)
tokenizer.fit_on_texts(train_df["payload"])
VOCAB_SIZE = len(tokenizer.word_index) + 1

def vectorize_payloads(payloads):
    return pad_sequences(
        tokenizer.texts_to_sequences(payloads),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post",
    )

X_train = vectorize_payloads(train_df["payload"])
X_val = vectorize_payloads(val_df["payload"])
X_test = vectorize_payloads(test_df["payload"])
X_obfuscated = vectorize_payloads(obfuscated_df["payload"])

y_train = train_df["label"].to_numpy(dtype=np.int32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
y_test = test_df["label"].to_numpy(dtype=np.int32)
y_obfuscated = obfuscated_df["label"].to_numpy(dtype=np.int32)

weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = {
    int(label): float(weight)
    for label, weight in zip(np.unique(y_train), weights)
}

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Obfuscated test: {len(obfuscated_df):,}")
print(f"Vocabulary size: {VOCAB_SIZE} | MAX_LEN: {MAX_LEN}")
print("Class weights:", class_weight_dict)


Train: 117,444 | Val: 13,050 | Test: 32,624
Obfuscated test: 150,000
Vocabulary size: 180 | MAX_LEN: 1024
Class weights: {0: 1.3983093225383973, 1: 0.7783005738975997}


## 3. Mô hình CNN và LSTM chạy song song

In [3]:
def build_parallel_cnn_lstm(vocab_size=VOCAB_SIZE, max_len=MAX_LEN, embedding_dim=EMBEDDING_DIM):
    inputs = Input(shape=(max_len,), dtype="int32", name="payload_tokens")
    embedded = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        name="char_embedding",
    )(inputs)

    # Cùng cấu hình CNN với CNN_LSTM.py; GlobalMaxPooling tạo vector để concatenate.
    cnn = Conv1D(128, 3, padding="same", activation="relu", name="cnn_conv_k3")(embedded)
    cnn = MaxPooling1D(pool_size=4, name="cnn_pool_1")(cnn)
    cnn = Conv1D(128, 5, padding="same", activation="relu", name="cnn_conv_k5")(cnn)
    cnn = MaxPooling1D(pool_size=4, name="cnn_pool_2")(cnn)
    cnn_vector = GlobalMaxPooling1D(name="cnn_global_max_pool")(cnn)

    # Cùng cấu hình LSTM với CNN_LSTM.py.
    lstm_vector = LSTM(128, name="lstm_context")(embedded)

    merged = Concatenate(name="parallel_feature_fusion")([cnn_vector, lstm_vector])
    dense = Dense(64, activation="relu", name="dense_classifier")(merged)
    dense = Dropout(0.3, name="dropout")(dense)
    outputs = Dense(1, activation="sigmoid", name="attack_probability")(dense)

    parallel_model = Model(inputs, outputs, name="Parallel_1D_CNN_LSTM_Web_Attack_Detector")
    parallel_model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            AUC(name="auc"),
            Precision(name="precision"),
            Recall(name="recall"),
        ],
    )
    return parallel_model


model = build_parallel_cnn_lstm()
model.summary()


Model: "Parallel_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ payload_tokens      │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_embedding      │ (None, 1024, 64)  │     11,520 │ payload_tokens[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv_k3         │ (None, 1024, 128) │     24,704 │ char_embedding[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool_1          │ (None, 256, 128)  │          0 │ cnn_conv_k3[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv_k5         │ (None, 256, 128)  │     82,048 │ cnn_pool_1[0][0]  │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool_2          │ (None, 64, 128)   │          0 │ cnn_conv_k5[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_global_max_pool │ (None, 128)       │          0 │ cnn_pool_2[0][0]  │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_context (LSTM) │ (None, 128)       │     98,816 │ char_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ parallel_feature_f… │ (None, 256)       │          0 │ cnn_global_max_p… │
│ (Concatenate)       │                   │            │ lstm_context[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_classifier    │ (None, 64)        │     16,448 │ parallel_feature… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense_classifier… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attack_probability  │ (None, 1)         │         65 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 233,601 (912.50 KB)

 Trainable params: 233,601 (912.50 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Huấn luyện

Cấu hình được giữ giống pipeline `CNN_LSTM.py`: 50 epoch, batch size 128,
class weight cân bằng, EarlyStopping và ModelCheckpoint theo `val_loss`.


In [4]:
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = ARTIFACT_DIR / "best_parallel_cnn_lstm.keras"
LAST_MODEL_PATH = ARTIFACT_DIR / "last_parallel_cnn_lstm.keras"
TOKENIZER_PATH = ARTIFACT_DIR / "tokenizer.pkl"
RESULTS_PATH = ARTIFACT_DIR / "metadata_and_results.json"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

model.save(LAST_MODEL_PATH)
with TOKENIZER_PATH.open("wb") as file:
    pickle.dump(tokenizer, file)


Epoch 1/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9431 - auc: 0.9820 - loss: 0.1407 - precision: 0.9616 - recall: 0.9506
Epoch 1: val_loss improved from None to 0.04805, saving model to C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\best_parallel_cnn_lstm.keras

Epoch 1: finished saving model to C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\best_parallel_cnn_lstm.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 1457s 2s/step - accuracy: 0.9685 - auc: 0.9963 - loss: 0.0738 - precision: 0.9863 - recall: 0.9644 - val_accuracy: 0.9782 - val_auc: 0.9983 - val_loss: 0.0480 - val_precision: 0.9985 - val_recall: 0.9674
Epoch 2/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9795 - auc: 0.9983 - loss: 0.0418 - precision: 0.9973 - recall: 0.9707
Epoch 2: val_loss improved from 0.04805 to 0.04088, saving model to C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\best_parallel_cnn_ls

## 5. Đánh giá với cùng ngưỡng 0.5 như `CNN_LSTM.py`

In [5]:
model = tf.keras.models.load_model(BEST_MODEL_PATH)

def evaluate_model(model, X, y, name):
    probabilities = model.predict(X, batch_size=BATCH_SIZE, verbose=1).ravel()
    predictions = (probabilities >= 0.5).astype(np.int32)
    report = classification_report(
        y,
        predictions,
        target_names=["Normal (0)", "Attack (1)"],
        output_dict=True,
        zero_division=0,
    )
    result = {
        "accuracy": float(accuracy_score(y, predictions)),
        "auc_roc": float(roc_auc_score(y, probabilities)) if len(np.unique(y)) > 1 else None,
        "confusion_matrix": confusion_matrix(y, predictions).tolist(),
        "classification_report": report,
    }
    print(f"\n=== {name.upper()} ===")
    print(classification_report(
        y,
        predictions,
        target_names=["Normal (0)", "Attack (1)"],
        digits=4,
        zero_division=0,
    ))
    print("Confusion matrix:\n", np.asarray(result["confusion_matrix"]))
    print("AUC-ROC:", result["auc_roc"])
    return result


test_result = evaluate_model(model, X_test, y_test, "normal test")
obfuscated_result = evaluate_model(model, X_obfuscated, y_obfuscated, "obfuscated test")


255/255 ━━━━━━━━━━━━━━━━━━━━ 126s 492ms/step

=== NORMAL TEST ===
              precision    recall  f1-score   support

  Normal (0)     0.9624    0.9922    0.9771     11666
  Attack (1)     0.9956    0.9784    0.9869     20958

    accuracy                         0.9834     32624
   macro avg     0.9790    0.9853    0.9820     32624
weighted avg     0.9837    0.9834    0.9834     32624

Confusion matrix:
 [[11575    91]
 [  452 20506]]
AUC-ROC: 0.9988711759358314
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 671s 573ms/step

=== OBFUSCATED TEST ===
              precision    recall  f1-score   support

  Normal (0)     0.0000    0.0000    0.0000         0
  Attack (1)     1.0000    0.9980    0.9990    150000

    accuracy                         0.9980    150000
   macro avg     0.5000    0.4990    0.4995    150000
weighted avg     1.0000    0.9980    0.9990    150000

Confusion matrix:
 [[     0      0]
 [   298 149702]]
AUC-ROC: None


## 6. Lưu kết quả và so sánh trực tiếp

In [ ]:
results = preprocessing_metadata.copy()
results["model"] = {
    "name": model.name,
    "parameter_count": int(model.count_params()),
    "topology": "Shared Embedding -> [CNN branch || LSTM branch] -> Concatenate -> Dense -> Sigmoid",
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "cnn_filters": [128, 128],
    "cnn_kernel_sizes": [3, 5],
    "cnn_pool_sizes": [4, 4],
    "lstm_units": 128,
    "dense_units": 64,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "decision_threshold": 0.5,
    "class_weight": class_weight_dict,
}
results["training_history"] = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}
results["evaluation"] = {
    "test": test_result,
    "obfuscated_test": obfuscated_result,
}

with RESULTS_PATH.open("w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

baseline_candidates = [
    PROJECT_ROOT / "cnn_lstm" / "artifacts_cnn_lstm_rerun" / "metadata_and_results.json",
    PROJECT_ROOT / "cnn_lstm" / "artifacts" / "metadata_and_results.json",
    PROJECT_ROOT / "cnn_lstm" / "artifacts_cnn_lstm_smoke" / "metadata_and_results.json",
]
baseline_path = next((path for path in baseline_candidates if path.exists()), None)

if baseline_path is None:
    print("Đã lưu kết quả song song:", RESULTS_PATH)
    print("Chưa có kết quả CNN_LSTM baseline. Hãy chạy cnn_lstm/CNN_LSTM.ipynb rồi chạy lại cell này.")
else:
    with baseline_path.open(encoding="utf-8") as file:
        baseline_results = json.load(file)

    def get_attack_report(classification_report):
        for attack_label in ("Attack (1)", "1"):
            if attack_label in classification_report:
                return classification_report[attack_label]
        raise KeyError(f"Không tìm thấy nhãn attack trong report: {list(classification_report.keys())}")

    def comparison_row(name, payload, params=None):
        test = payload["evaluation"]["test"]
        obfu = payload["evaluation"]["obfuscated_test"]
        attack = get_attack_report(test["classification_report"])
        obfu_attack = get_attack_report(obfu["classification_report"])
        return {
            "Model": name,
            "Params": params,
            "Test Accuracy": test["accuracy"],
            "Test AUC": test["auc_roc"],
            "Test Attack Recall": attack["recall"],
            "Test Attack F1": attack["f1-score"],
            "Obfuscated Recall": obfu_attack["recall"],
            "Obfuscated F1": obfu_attack["f1-score"],
            "Obfuscated FN": obfu["confusion_matrix"][1][0],
        }

    comparison = pd.DataFrame([
        comparison_row("CNN-LSTM tuần tự", baseline_results),
        comparison_row("CNN || LSTM song song", results, model.count_params()),
    ])
    display(comparison)
    print("Baseline:", baseline_path)
    print("Parallel:", RESULTS_PATH)


KeyError: 'Attack (1)'

## Nguyên tắc kết luận

Chỉ kết luận topology song song tốt hơn hoặc kém hơn khi hai lần chạy dùng cùng dữ liệu,
seed, tokenizer, `MAX_LEN`, siêu tham số huấn luyện và ngưỡng 0.5. Ưu tiên xem
Attack Recall/F1, số false negative trên tập obfuscated và thời gian/parameter count;
không chỉ nhìn accuracy.
